# Working Backwards Experiments
This notebook recomputes the latency distribution experiment and persists the Population Stability Index (PSI) to a reproducible artifact so downstream reports can consume a numeric value.


In [ ]:
from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

ARTIFACT_PATH = Path("docs/notebooks/artifacts/latency_summary.json")

baseline_latency = {
    "0-200ms": 0.42,
    "200-400ms": 0.37,
    "400ms+": 0.21,
}
current_latency = {
    "0-200ms": 0.40,
    "200-400ms": 0.38,
    "400ms+": 0.22,
}
PSI_THRESHOLD = 0.2

existing_artifact: dict[str, object] | None = None
if ARTIFACT_PATH.exists():
    with ARTIFACT_PATH.open("r", encoding="utf-8") as fp:
        existing_artifact = json.load(fp)

def calculate_psi(baseline: dict[str, float], current: dict[str, float], *, epsilon: float = 1e-6) -> float:
    """Compute PSI with epsilon guards for empty buckets."""
    psi = 0.0
    for bucket, baseline_share in baseline.items():
        current_share = current.get(bucket, epsilon)
        baseline_adj = baseline_share if baseline_share > 0 else epsilon
        current_adj = current_share if current_share > 0 else epsilon
        psi += (current_adj - baseline_adj) * math.log(current_adj / baseline_adj)
    return psi

psi_value = round(calculate_psi(baseline_latency, current_latency), 6)
psi_value


In [ ]:
def isoformat_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

psi_status = "within_threshold" if psi_value <= PSI_THRESHOLD else "breach"
content_without_hash = {
    "schema_version": 1,
    "experiment_id": "working_backwards_experiments",
    "generated_at": isoformat_now(),
    "baseline_latency_share": baseline_latency,
    "current_latency_share": current_latency,
    "metrics": {
        "psi": psi_value,
        "psi_threshold": PSI_THRESHOLD,
        "status": psi_status,
    },
}

if existing_artifact:
    existing_wo_hash = {k: v for k, v in existing_artifact.items() if k != "artifact_sha256"}
    existing_wo_ts = dict(existing_wo_hash)
    existing_wo_ts.pop("generated_at", None)

    new_wo_ts = dict(content_without_hash)
    new_wo_ts.pop("generated_at", None)

    if existing_wo_ts == new_wo_ts:
        content_without_hash["generated_at"] = existing_artifact.get("generated_at", content_without_hash["generated_at"])

canonical_bytes = json.dumps(content_without_hash, sort_keys=True, separators=(",", ":")).encode("utf-8")
artifact_sha256 = sha256(canonical_bytes).hexdigest()
latency_summary = {**content_without_hash, "artifact_sha256": artifact_sha256}

ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
with ARTIFACT_PATH.open("w", encoding="utf-8") as fp:
    json.dump(latency_summary, fp, indent=2, sort_keys=True)
    fp.write("\n")

psi_value, artifact_sha256, latency_summary
